In [ ]:
import math
import os
import zipfile
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

SHOW_FIGURES  = True
VOL_WINDOW    = 60
EXPORT_FORMAT = "html"
EXPORT_DIR    = "greeks_export"
EXPORT_ZIP    = "greeks_export.zip"


def _detect_renderer() -> str:

    try:
        from google.colab import output
        return "colab"
    except ImportError:
        pass
    try:
        shell = get_ipython().__class__.__name__
        if shell == "ZMQInteractiveShell":
            return "notebook_connected"
        if shell == "TerminalInteractiveShell":
            return "browser"
    except NameError:
        pass
    return "browser"


_RENDERER = _detect_renderer()
pio.renderers.default = _RENDERER
print(f"  Plotly renderer: {_RENDERER}")

#CDF function (Abramowiz & Stegun)

_AS_P = 0.2316419
_AS_C = (0.319381530,
         -0.356563782,
          1.781477937,
         -1.821255978,
          1.330274429)


def normal_cdf(x: float) -> float:

    if x < -8.0: return 0.0
    if x >  8.0: return 1.0
    if x >= 0.0:
        k    = 1.0 / (1.0 + _AS_P * x)
        poly = k * (_AS_C[0] + k * (_AS_C[1] + k * (_AS_C[2] + k * (_AS_C[3] + k * _AS_C[4]))))
        pdf  = math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)
        return 1.0 - pdf * poly
    return 1.0 - normal_cdf(-x)


def norm_pdf(x: float) -> float:

    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)


# BSM Pricer  (scalar, pure Python)

def bsm_price(S: float, K: float, tau: float, r: float, q: float,
              sigma: float, option_type: str = "call") -> float:

    if tau < 0:
        raise ValueError(f"tau must be >= 0, got {tau}")
    if sigma < 0:
        raise ValueError(f"sigma must be >= 0, got {sigma}")

    tau_y = tau / 365.0
    if tau_y == 0.0:
        return max(0.0, S - K) if option_type == "call" else max(0.0, K - S)

    sqrt_tau = math.sqrt(tau_y)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma * sigma) * tau_y) / (sigma * sqrt_tau)
    d2 = d1 - sigma * sqrt_tau

    if option_type == "call":
        return (S * math.exp(-q * tau_y) * normal_cdf(d1)
                - K * math.exp(-r * tau_y) * normal_cdf(d2))
    elif option_type == "put":
        return (K * math.exp(-r * tau_y) * normal_cdf(-d2)
                - S * math.exp(-q * tau_y) * normal_cdf(-d1))
    else:
        raise ValueError(f"option_type must be 'call' or 'put', got '{option_type}'")


# Greeks:FINITE DIFFERENCE METHOD

_H_S     = 0.01
_H_SIGMA = 0.0001
_H_TAU   = 1.0
_H_R     = 0.0001


def fd_greeks(S: float, K: float, tau: float, r: float, q: float,
              sigma: float, option_type: str = "call") -> tuple:

    p0 = bsm_price(S, K, tau, r, q, sigma, option_type)

    h_s   = _H_S * S
    p_up  = bsm_price(S + h_s, K, tau, r, q, sigma, option_type)
    p_dn  = bsm_price(S - h_s, K, tau, r, q, sigma, option_type)
    delta = (p_up - p_dn) / (2.0 * h_s)

    gamma = (p_up - 2.0 * p0 + p_dn) / (h_s * h_s)

    p_vs  = bsm_price(S, K, tau, r, q, sigma + _H_SIGMA, option_type)
    p_vd  = bsm_price(S, K, tau, r, q, sigma - _H_SIGMA, option_type)
    vega  = (p_vs - p_vd) / (2.0 * _H_SIGMA) * 0.01

    if tau > _H_TAU:
        p_th = bsm_price(S, K, tau - _H_TAU, r, q, sigma, option_type)
        theta = (p_th - p0) / _H_TAU
    else:
        theta = 0.0

    p_ru = bsm_price(S, K, tau, r + _H_R, q, sigma, option_type)
    p_rd = bsm_price(S, K, tau, r - _H_R, q, sigma, option_type)
    rho  = (p_ru - p_rd) / (2.0 * _H_R) * 0.01

    return p0, delta, gamma, vega, theta, rho


# Implied Volatility solver:Newton-Raphson

def implied_vol(market_price: float, S: float, K: float, tau: float,
                r: float, q: float, option_type: str = "call",
                tol: float = 1e-7, max_iter: int = 200) -> float:

    sigma_iv = 0.30
    for _ in range(max_iter):
        p, _, _, vega_pp, _, _ = fd_greeks(S, K, tau, r, q, sigma_iv, option_type)
        diff = p - market_price
        if abs(diff) < tol:
            return sigma_iv
        raw_vega = vega_pp / 0.01
        if abs(raw_vega) < 1e-12:
            break
        sigma_iv -= diff / raw_vega
        sigma_iv  = max(1e-6, min(sigma_iv, 10.0))
    raise ValueError(f"IV did not converge for market_price={market_price:.6f}")


# Market data layer

def fetch_market_data(ticker: str, vol_window: int = VOL_WINDOW):

    end   = pd.Timestamp.today()
    start = end - pd.Timedelta(days=vol_window + 60)
    raw   = yf.download(ticker,
                        start=start.strftime("%Y-%m-%d"),
                        end=end.strftime("%Y-%m-%d"),
                        progress=False, auto_adjust=True)
    if raw.empty:
        raise ValueError(f"No data found for ticker '{ticker}'.")
    closes = raw["Close"].squeeze().dropna()
    prices = list(closes.values)
    if len(prices) < 5:
        raise ValueError(f"Not enough price history for '{ticker}'.")
    spot  = float(prices[-1])

    tail  = prices[-(vol_window + 1):] if len(prices) > vol_window + 1 else prices
    lrets = [math.log(tail[i] / tail[i - 1]) for i in range(1, len(tail))]
    m     = sum(lrets) / len(lrets)
    var   = sum((x - m) ** 2 for x in lrets) / (len(lrets) - 1)
    sigma_real = math.sqrt(var * 252)

    risk_free = 0.04
    try:
        irx = yf.download("^IRX", period="5d", progress=False, auto_adjust=True)
        if not irx.empty:
            risk_free = float(irx["Close"].dropna().iloc[-1]) / 100.0
    except Exception:
        pass

    div_yield = 0.0
    try:
        tk   = yf.Ticker(ticker)
        divs = tk.dividends
        if divs is not None and not divs.empty:
            cutoff    = pd.Timestamp.today(tz=divs.index.tz) - pd.DateOffset(years=1)
            last_year = divs[divs.index >= cutoff]
            if not last_year.empty and spot > 0:
                div_yield = float(last_year.sum()) / spot
    except Exception:
        pass

    return spot, risk_free, div_yield, sigma_real


# Interactive configuration helper

def _ask(prompt, cast=str, default=None, valid=None):
    while True:
        raw = input(prompt).strip()
        if raw == "" and default is not None:
            return default
        try:
            val = cast(raw)
        except (ValueError, TypeError):
            print("      Invalid input."); continue
        if valid is not None and val not in valid:
            print(f"      Choose one of: {' / '.join(map(str, valid))}"); continue
        return val


def configure_market() -> dict:
    print("\n" + "=" * 45)
    print("  MARKET CONFIGURATION")
    print("=" * 45)

    use_real = _ask("Use a REAL stock? [y/n] (default n): ",
                    str, default="n", valid={"y", "n"}) == "y"

    asset_name = "CUSTOM"
    spot_real = r_real = q_real = sigma_real = None

    if use_real:
        ticker = _ask("Ticker (e.g. AAPL): ", str, default="AAPL").upper()
        print(f"    Fetching {ticker} ", end="", flush=True)
        spot_real, r_real, q_real, sigma_real = fetch_market_data(ticker)
        print("YES")
        asset_name = ticker
        print(f"    Spot        : {spot_real:.4f}")
        print(f"    Risk-free   : {r_real*100:.3f}%  [^IRX]")
        print(f"    Div yield   : {q_real*100:.3f}%  [trailing 12M]")
        print(f"    Realized vol: {sigma_real*100:.2f}%  [{VOL_WINDOW}d close-to-close]")

    if use_real:
        S_cfg = _ask(f"Spot S:use real {spot_real:.4f}? [y/n] (default y): ",
                     str, default="y", valid={"y", "n"})
        S_val = spot_real if S_cfg == "y" else _ask("  Custom S: ", float)
    else:
        S_val = _ask("Spot S (default 105.0): ", float, default=105.0)

    if use_real:
        ratio     = _ask("K as ratio of spot (e.g. 1.05 = 5% OTM call, default 1.00): ",
                         float, default=1.00)
        k_suggest = round(S_val * ratio, 2)
        K_val     = _ask(f"Strike K (suggested {k_suggest:.2f}, editable): ",
                         float, default=k_suggest)
    else:
        K_val = _ask("Strike K (default 105.0): ", float, default=105.0)

    tau_val = _ask("Days to expiry tau (default 90): ", int, default=90)
    if tau_val < 1:
        print("    ATTENTION!  tau must be >= 1 day:setting to 1.")
        tau_val = 1

    if use_real:
        r_cfg = _ask(f"Rate r:use real {r_real:.4f}? [y/n] (default y): ",
                     str, default="y", valid={"y", "n"})
        r_val = r_real if r_cfg == "y" else _ask("  Custom r: ", float)
    else:
        r_val = _ask("Risk-free r (default 0.03): ", float, default=0.03)

    if use_real:
        q_cfg = _ask(f"Div yield q:use real {q_real:.4f}? [y/n] (default y): ",
                     str, default="y", valid={"y", "n"})
        q_val = q_real if q_cfg == "y" else _ask("  Custom q: ", float)
    else:
        q_val = _ask("Dividend yield q (default 0.015): ", float, default=0.015)

    if use_real:
        s_cfg = _ask(f"Vol sigma:use real {sigma_real:.4f}? [y/n] (default y): ",
                     str, default="y", valid={"y", "n"})
        sigma_val = sigma_real if s_cfg == "y" else _ask("  Custom sigma: ", float)
    else:
        sigma_val = _ask("Volatility sigma (default 0.27): ", float, default=0.27)

    return dict(S=S_val, K=K_val, tau=tau_val, r=r_val, q=q_val,
                sigma=sigma_val, asset_name=asset_name)


try:
    _cfg = configure_market()
    S          = _cfg["S"]
    K          = _cfg["K"]
    tau        = _cfg["tau"]
    r          = _cfg["r"]
    q          = _cfg["q"]
    sigma      = _cfg["sigma"]
    ASSET_NAME = _cfg["asset_name"]
except (EOFError, OSError):
    print("\n  ATTENTION! Non-interactive environment detected:using default parameters.")
    S     = 105.0
    K     = 105.0
    tau   = 90
    r     = 0.03
    q     = 0.015
    sigma = 0.27
    ASSET_NAME = "CUSTOM"


for option_type in ("call", "put"):
    price = bsm_price(S, K, tau, r, q, sigma, option_type)
    label = f"{ASSET_NAME} European {option_type.capitalize()} Option (BSM)"
    print(f"=== {label} ===")
    print(f"S={S:.5f}, K={K:.5f}, tau={tau} days ({tau/365:.5f} yrs), "
          f"r={r:.5f}, q={q:.5f}, sigma={sigma:.5f}")
    print("-" * 45)
    print(f"Price : {price:.6f}\n")


print("\n=== Greeks (Finite Difference) ===")
for option_type in ("call", "put"):
    price, delta, gamma, vega, theta, rho = fd_greeks(
        S, K, tau, r, q, sigma, option_type)
    label = f"{ASSET_NAME} European {option_type.capitalize()} Option"
    print(f"\n  {label}")
    print(f"  S={S:.5f}, K={K:.5f}, tau={tau}d, r={r:.5f}, q={q:.5f}, sigma={sigma:.5f}")
    print(f"  Price  : {price:.6f}")
    print(f"  Delta  : {delta:.6f}   ($/$ spot move)")
    print(f"  Gamma  : {gamma:.6f}   (Delta per $ spot move)")
    print(f"  Vega   : {vega:.6f}   ($ per 1 pp sigma)")
    print(f"  Theta  : {theta:.6f}   ($ per calendar day)")
    print(f"  Rho    : {rho:.6f}   ($ per 1 pp r)")


_demo_price = bsm_price(S, K, tau, r, q, sigma, "call")
_demo_iv    = implied_vol(_demo_price, S, K, tau, r, q, "call")
print(f"\n[IV demo]  call price={_demo_price:.6f}  ->  IV={_demo_iv*100:.4f}%  "
      f"(input sigma={sigma*100:.4f}%)")


COLORSCALE_MAP = {
    "delta": "Viridis",
    "gamma": "Plasma",
    "vega" : "Cividis",
    "theta": "RdBu",
    "rho"  : "Magma",
}

GREEK_IDX = {"price": 0, "delta": 1, "gamma": 2, "vega": 3, "theta": 4, "rho": 5}


def build_grids(width_S: float = 0.6, n: int = 60):

    S_min    = max(1.0, S * (1.0 - width_S))
    S_max    = S * (1.0 + width_S)
    S_vals   = np.linspace(S_min, S_max, n)
    tau_vals = np.linspace(1.0, tau, n)
    return S_vals, tau_vals, S_min, S_max


def compute_surface(greek: str, option_type: str,
                    S_vals, tau_vals) -> np.ndarray:

    n_tau = len(tau_vals)
    n_S   = len(S_vals)
    idx   = GREEK_IDX[greek]

    Z = [[0.0] * n_S for _ in range(n_tau)]

    for i, tau_i in enumerate(tau_vals):
        for j, S_j in enumerate(S_vals):
            greeks_tuple = fd_greeks(float(S_j), K, float(tau_i), r, q, sigma, option_type)
            Z[i][j] = greeks_tuple[idx]

    return np.array(Z, dtype=float)


def _display_fig(fig):

    if SHOW_FIGURES:
        fig.show(renderer=_RENDERER)


def plot_surface_interactive(S_vals, tau_vals, Z, title, colorscale="Viridis"):

    fig = go.Figure(data=[go.Surface(
        x=S_vals,
        y=tau_vals,
        z=Z,
        colorscale=colorscale,
        colorbar=dict(title="Value", thickness=15, len=0.7),
        hovertemplate=(
            "S: %{x:.2f}<br>"
            "Days to expiry: %{y:.1f}<br>"
            "Value: %{z:.4f}<extra></extra>"
        ),
    )])
    fig.update_layout(
        title=dict(text=title, font=dict(size=14, family="monospace")),
        scene=dict(
            xaxis_title="Underlying price  S",
            yaxis_title="Days to expiry  tau",
            zaxis_title="Greek value",
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            aspectratio=dict(x=1.2, y=1.2, z=0.8),
            xaxis=dict(showbackground=True, backgroundcolor="rgb(240,240,245)"),
            yaxis=dict(showbackground=True, backgroundcolor="rgb(235,240,245)"),
            zaxis=dict(showbackground=True, backgroundcolor="rgb(240,245,245)"),
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        width=850, height=620,
    )
    _display_fig(fig)
    return fig


def plot_all_greeks_interactive(option_type: str = "call", n: int = 30):
    S_vals, tau_vals, S_min, S_max = build_grids(n=n)
    for g in COLORSCALE_MAP:
        print(f"  Computing {g} surface ({option_type}) ", end="", flush=True)
        Z = compute_surface(g, option_type, S_vals, tau_vals)
        print(" done")
        title = (f"{ASSET_NAME} | {g.capitalize()} | {option_type.capitalize()} | "
                 f"S in [{S_min:.2f}, {S_max:.2f}], sigma={sigma:.5f}, "
                 f"r={r:.5f}, q={q:.5f}, K={K:.5f}, tau={tau} days")
        plot_surface_interactive(S_vals, tau_vals, Z, title, COLORSCALE_MAP[g])


def plot_dashboard_greeks(option_type: str = "call", n: int = 25, show: bool = True):
    S_vals, tau_vals, S_min, S_max = build_grids(n=n)
    greeks = list(COLORSCALE_MAP.keys())
    specs = [
        [{"type": "surface"}, {"type": "surface"}, {"type": "surface"}],
        [{"type": "surface"}, {"type": "surface"}, {"type": "surface"}],
    ]
    fig = make_subplots(rows=2, cols=3, specs=specs,
                        subplot_titles=[g.capitalize() for g in greeks] + [""])
    for idx, g in enumerate(greeks):
        print(f"    [{g}] computing ", end="", flush=True)
        Z = compute_surface(g, option_type, S_vals, tau_vals)
        print(" ok")
        row, col = divmod(idx, 3)
        fig.add_trace(go.Surface(
            x=S_vals, y=tau_vals, z=Z,
            colorscale=COLORSCALE_MAP[g], showscale=False,
            hovertemplate=(f"<b>{g.capitalize()}</b><br>"
                           "S: %{x:.2f}<br>Days: %{y:.1f}<br>"
                           "Value: %{z:.4f}<extra></extra>"),
        ), row=row + 1, col=col + 1)

    for i in range(1, 6):
        scene_key = "scene" if i == 1 else f"scene{i}"
        fig.update_layout(**{
            scene_key: dict(
                xaxis_title="S",
                yaxis_title="tau (days)",
                zaxis_title=greeks[i - 1].capitalize() if i <= len(greeks) else "",
            )
        })

    fig.update_layout(
        title=dict(
            text=(f"{ASSET_NAME} | Greeks Dashboard | {option_type.capitalize()} | "
                  f"S in [{S_min:.2f}, {S_max:.2f}], K={K:.5f}, sigma={sigma:.5f}, "
                  f"r={r:.5f}, q={q:.5f}, tau={tau} days  [Finite Difference]"),
            font=dict(size=14, family="monospace"),
        ),
        height=800, width=1200, margin=dict(l=0, r=0, t=70, b=0),
    )
    if show:
        _display_fig(fig)
    return fig


def _ensure_export_dir():
    os.makedirs(EXPORT_DIR, exist_ok=True)


def save_figure(fig, filename_base: str) -> str:
    _ensure_export_dir()
    path = os.path.join(EXPORT_DIR, f"{filename_base}.{EXPORT_FORMAT}")
    fig.write_html(path, include_plotlyjs="cdn")
    print(f"    Saved: {path}")
    return path


def export_all_to_zip(saved_paths: list, zip_name: str = None) -> str:
    zip_name = zip_name or EXPORT_ZIP
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in saved_paths:
            zf.write(p, arcname=os.path.basename(p))
    print(f"\n  ZIP created: {zip_name}  ({len(saved_paths)} files)")
    return zip_name


def export_all_greeks(options=("call", "put"),
                      greeks=("delta", "gamma", "vega", "theta", "rho"),
                      include_dashboard: bool = True,
                      make_zip: bool = True,
                      show: bool = False,
                      n: int = 30) -> list:

    S_vals, tau_vals, S_min, S_max = build_grids(n=n)
    saved = []
    print(f"\n  Exporting:format: {EXPORT_FORMAT.upper()}")
    print(f"  Output folder: {EXPORT_DIR}/\n")
    for opt in options:
        for g in greeks:
            print(f"  Computing {g} ({opt}) ", end="", flush=True)
            Z = compute_surface(g, opt, S_vals, tau_vals)
            print(" done")
            title = (f"{ASSET_NAME} | {g.capitalize()} | {opt.capitalize()} | "
                     f"S in [{S_min:.2f}, {S_max:.2f}], sigma={sigma:.5f}, "
                     f"r={r:.5f}, q={q:.5f}, K={K:.5f}, tau={tau} days  [FD]")
            fig = plot_surface_interactive(S_vals, tau_vals, Z, title, COLORSCALE_MAP[g])
            if show:
                _display_fig(fig)
            saved.append(save_figure(fig, f"{ASSET_NAME}_{g}_{opt}"))
    if include_dashboard:
        for opt in options:
            print(f"\n  -> Dashboard {opt} ")
            fig = plot_dashboard_greeks(opt, n=min(n, 20), show=show)
            saved.append(save_figure(fig, f"{ASSET_NAME}_dashboard_{opt}"))
    if make_zip:
        export_all_to_zip(saved)
    print(f"\n  Export complete:{len(saved)} files saved.")
    return saved


print("\n  [FD Surfaces] Computing individual Greek surfaces ")
plot_all_greeks_interactive(option_type="call", n=25)
plot_all_greeks_interactive(option_type="put",  n=25)

print("\n  [FD Surfaces] Exporting all Greeks ")
export_all_greeks(n=25)


# HISTORICAL HEDGE SIMULATION

if ASSET_NAME == "CUSTOM":
    print("\n" + "=" * 60)
    print("  Custom underlying:no historical path available.")
    print("  Hedge simulation needs a REAL stock.  Stopping here.")
    print("=" * 60)

else:

    def _sim_greeks(S_: float, K_: float, tau_y_: float,
                    r_: float, q_: float, sigma_: float,
                    opt_type: str) -> tuple:

        tau_days = tau_y_ * 365.0
        p, d, g, *_ = fd_greeks(S_, K_, tau_days, r_, q_, sigma_, opt_type)
        return p, d, g


    def _sim_rolling_vol(prices_all: list, sim_start_idx: int,
                         sim_day: int, vol_window: int,
                         fallback_sigma: float) -> float:
        end_idx   = sim_start_idx + sim_day
        start_idx = max(end_idx - vol_window, 0)
        chunk     = prices_all[start_idx: end_idx + 1]
        if len(chunk) < 3:
            return fallback_sigma
        lrets = [math.log(chunk[i] / chunk[i - 1]) for i in range(1, len(chunk))]
        m = sum(lrets) / len(lrets)
        v = sum((x - m) ** 2 for x in lrets) / (len(lrets) - 1)
        return math.sqrt(v * 252)


    def _sim_round_to_lot(qty: float, lot: int = 1) -> int:
        return int(round(qty / lot) * lot)


    def _sim_auto_hedge_type(pos_dir: str, pos_opt: str) -> tuple:
        return {
            ("short", "call"): ("put",  "long"),
            ("short", "put"):  ("call", "long"),
            ("long",  "call"): ("put",  "short"),
            ("long",  "put"):  ("call", "short"),
        }[(pos_dir, pos_opt)]


    def _sim_compute_hedge(port_d, port_g, dh_ps, gh_ps, mult):
        gh_total = mult * gh_ps
        if abs(gh_total) < 1e-14:
            raise ValueError("Hedge option gamma near zero:choose different strike.")
        w_h = _sim_round_to_lot(-port_g / gh_total)
        w_s = _sim_round_to_lot(-(port_d + w_h * mult * dh_ps))
        return w_h, w_s


    def _sim_run_simulation(cfg: dict) -> list:
        mode       = cfg["mode"]
        h_opt      = cfg["h_opt"]
        pos_sign   = cfg["pos_sign"]
        pos_qty    = cfg["pos_qty"]
        r_         = cfg["risk_free"]
        q_         = cfg["div_yield"]
        mult       = cfg["multiplier"]
        vol_window = cfg["vol_window"]
        prices_all = cfg["prices_all"]
        dates_all  = cfg["dates_all"]
        fb_sigma   = cfg["sigma"]
        dte        = cfg["dte"]
        hedge_days = cfg["hedge_days"]

        sim_start_idx = len(prices_all) - hedge_days
        prices = prices_all[sim_start_idx:]
        dates  = dates_all[sim_start_idx:]

        DIV = "=" * 76
        print(f"\n{DIV}")
        print(f"  SIMULATION  ·  {cfg['ticker']}  ·  {mode.upper()}")
        print(f"  {len(prices)} days  ({dates[0].date()} -> {dates[-1].date()})")
        print(f"  Multiplier: {mult}  |  Vol window: {vol_window}-day rolling")
        print(f"  Greeks via: FINITE DIFFERENCE")
        print(DIV)

        S0      = prices[0]
        tau0_y  = dte / 365.0
        sigma_  = _sim_rolling_vol(prices_all, sim_start_idx, 0, vol_window, fb_sigma)

        p0, d0, g0 = _sim_greeks(S0, cfg["pos_K"], tau0_y, r_, q_, sigma_, cfg["opt_type"])
        port_d = pos_sign * pos_qty * mult * d0
        port_g = pos_sign * pos_qty * mult * g0

        if h_opt:
            ph0, dh0, gh0 = _sim_greeks(S0, h_opt["K"], tau0_y, r_, q_, sigma_, h_opt["opt_type"])
            w_h, w_s = _sim_compute_hedge(port_d, port_g, dh0, gh0, mult)
        else:
            ph0 = dh0 = gh0 = 0.0
            w_h = 0
            w_s = _sim_round_to_lot(-port_d)

        cash = pos_sign * (-1) * pos_qty * mult * p0
        if h_opt:
            cash -= w_h * mult * ph0
        cash -= w_s * S0

        net_d0 = port_d + (w_h * mult * dh0 if h_opt else 0.0) + w_s
        net_g0 = port_g + (w_h * mult * gh0 if h_opt else 0.0)

        print(f"\nDay 0 setup:")
        print(f"  Spot        : ${S0:.4f}  |  sigma={sigma_*100:.2f}%  "
              f"|  r={r_*100:.3f}%  |  q={q_*100:.3f}%")
        print(f"  Position    : {cfg['pos_dir'].upper()} {pos_qty} {cfg['opt_type']} ctr  "
              f"K={cfg['pos_K']:.2f}  ->  ${p0:.4f}/sh  Delta={d0:.6f}  Gamma={g0:.6f}")
        print(f"  Port Delta  : {port_d:+.4f}  |  Port Gamma: {port_g:+.6f}")
        if h_opt:
            print(f"  Hedge option: {w_h:+.0f} contracts  [{h_opt['label']}]")
        print(f"  Stock       : {w_s:+.0f} shares  @  ${S0:.4f}")
        print(f"  Cash        : ${cash:,.4f}")
        print(f"  Net Delta   : {net_d0:+.4f}")
        print(f"  Net Gamma   : {net_g0:+.6f}")

        cash_unhedged = pos_sign * (-1) * pos_qty * mult * p0

        history = [{
            "day": 0, "date": dates[0].date(), "spot": S0, "sigma": sigma_,
            "tau_y": tau0_y, "pos_price": p0, "h_price": ph0,
            "w_h_start": w_h, "w_h_end": w_h, "dw_h": w_h,
            "opt_cost": -(w_h * mult * ph0),
            "w_s_start": w_s, "w_s_end": w_s, "dw_s": w_s,
            "stock_cost": -(w_s * S0),
            "net_delta": net_d0, "net_gamma": net_g0,
            "net_delta_start": net_d0, "net_gamma_start": net_g0,
            "net_delta_end": net_d0, "net_gamma_end": net_g0,
            "cash": cash, "mtm_pnl": None, "unhedged_pnl": None,
            "pnl_delta": None, "pnl_gamma": None,
            "pnl_theta": None, "pnl_residual": None,
            "rebalanced": False, "flag": "OPEN",
        }]

        for i in range(1, len(prices)):
            S_    = prices[i]
            date  = dates[i]
            dS    = S_ - prices[i - 1]

            remaining_days = dte - i
            if remaining_days < 1:
                print(f"\n  ATTENTION! Option expired after {i} trading days:simulation ends.")
                break

            tau_y  = remaining_days / 365.0
            sigma_ = _sim_rolling_vol(prices_all, sim_start_idx, i, vol_window, fb_sigma)

            pm, dm, gm = _sim_greeks(S_, cfg["pos_K"], tau_y, r_, q_, sigma_, cfg["opt_type"])
            port_d = pos_sign * pos_qty * mult * dm
            port_g = pos_sign * pos_qty * mult * gm

            w_h_start, w_s_start = w_h, w_s

            if h_opt:
                ph_c, dh_c, gh_c = _sim_greeks(S_, h_opt["K"], tau_y, r_, q_, sigma_, h_opt["opt_type"])
            else:
                ph_c = dh_c = gh_c = 0.0

            dw_h = dw_s = 0
            opt_cost = stock_cost = 0.0
            rebalanced = False
            flag = ""

            if i % cfg["rebal_n"] == 0 and tau_y > 1e-4:
                if h_opt:
                    if abs(mult * gh_c) < 1e-14:
                        nw_s = _sim_round_to_lot(-(port_d + w_h * mult * dh_c))
                        dw_s = nw_s - w_s
                        stock_cost = -(dw_s * S_)
                        cash += stock_cost
                        w_s = nw_s
                        flag = "Gamma-FLAT"
                    else:
                        nw_h, nw_s = _sim_compute_hedge(port_d, port_g, dh_c, gh_c, mult)
                        dw_h = nw_h - w_h
                        dw_s = nw_s - w_s
                        opt_cost   = -(dw_h * mult * ph_c)
                        stock_cost = -(dw_s * S_)
                        cash += opt_cost + stock_cost
                        w_h = nw_h
                        w_s = nw_s
                        flag = "REBAL"
                else:
                    nw_s = _sim_round_to_lot(-port_d)
                    dw_s = nw_s - w_s
                    stock_cost = -(dw_s * S_)
                    cash += stock_cost
                    w_s = nw_s
                    flag = "REBAL"
                rebalanced = True

            if h_opt:
                net_d = port_d + w_h * mult * dh_c + w_s
                net_g = port_g + w_h * mult * gh_c
                mtm   = (pos_sign * pos_qty * mult * pm
                         + w_h * mult * ph_c + w_s * S_ + cash)
                net_d_start = port_d + w_h_start * mult * dh_c + w_s_start
                net_g_start = port_g + w_h_start * mult * gh_c
            else:
                net_d = port_d + w_s
                net_g = port_g
                mtm   = pos_sign * pos_qty * mult * pm + w_s * S_ + cash
                net_d_start = port_d + w_s_start
                net_g_start = port_g

            unhedged_mtm = pos_sign * pos_qty * mult * pm + cash_unhedged

            prev_row   = history[-1]
            prev_d     = prev_row["net_delta"]
            prev_g     = prev_row["net_gamma"]
            pnl_delta  = prev_d * dS
            pnl_gamma  = 0.5 * prev_g * dS ** 2

            _, _, _, _, th_prev, _ = fd_greeks(
                prices[i - 1], cfg["pos_K"],
                float(dte - (i - 1)),
                r_, q_, sigma_, cfg["opt_type"])
            pnl_theta    = pos_sign * pos_qty * mult * th_prev * 1.0
            pnl_total    = ((mtm - prev_row["mtm_pnl"])
                            if prev_row["mtm_pnl"] is not None else 0.0)
            pnl_residual = pnl_total - pnl_delta - pnl_gamma - pnl_theta

            history.append({
                "day": i, "date": date.date(), "spot": S_, "sigma": sigma_,
                "tau_y": tau_y, "pos_price": pm, "h_price": ph_c,
                "w_h_start": w_h_start, "w_h_end": w_h, "dw_h": dw_h,
                "opt_cost": opt_cost,
                "w_s_start": w_s_start, "w_s_end": w_s, "dw_s": dw_s,
                "stock_cost": stock_cost,
                "net_delta": net_d, "net_gamma": net_g,
                "net_delta_start": net_d_start, "net_gamma_start": net_g_start,
                "net_delta_end": net_d, "net_gamma_end": net_g,
                "cash": cash, "mtm_pnl": mtm, "unhedged_pnl": unhedged_mtm,
                "pnl_delta": pnl_delta, "pnl_gamma": pnl_gamma,
                "pnl_theta": pnl_theta, "pnl_residual": pnl_residual,
                "rebalanced": rebalanced, "flag": flag,
            })

        DIV = "=" * 76
        print(f"\n{DIV}")
        print("  NET Delta / Gamma : START vs END OF DAY")
        print(DIV)
        Wd, Wdate, Wcol = 4, 12, 16
        hdr = (f"{'Day':>{Wd}} {'Date':>{Wdate}}  "
               f"{'starting delta':>{Wcol}} {'ending delta':>{Wcol}} "
               f"{'starting gamma':>{Wcol}} {'ending gamma':>{Wcol}}")
        print(hdr)
        print("-" * len(hdr))
        for row in history:
            print(f"{row['day']:>{Wd}} {str(row['date']):>{Wdate}}  "
                  f"{row['net_delta_start']:>{Wcol}.4f} "
                  f"{row['net_delta_end']:>{Wcol}.4f} "
                  f"{row['net_gamma_start']:>{Wcol}.6f} "
                  f"{row['net_gamma_end']:>{Wcol}.6f}")
        print("-" * len(hdr))

        fin = history[-1]; ini = history[0]
        n_rb = sum(1 for h in history if h["rebalanced"])
        pnl  = [h["mtm_pnl"] for h in history if h["mtm_pnl"] is not None]

        print(f"\n{DIV}")
        print(f"  FINAL SUMMARY  ·  {mode.upper()}  ·  {cfg['ticker']}")
        print(DIV)
        print(f"  Trading days          : {len(history)}")
        print(f"  Rebalances            : {n_rb}  (every {cfg['rebal_n']} day(s))")
        print(f"  Initial spot          : ${ini['spot']:.4f}")
        print(f"  Final   spot          : ${fin['spot']:.4f}  "
              f"({(fin['spot']/ini['spot']-1)*100:+.2f}%)")
        print(f"  Position              : {cfg['pos_dir'].upper()} {pos_qty} "
              f"{cfg['opt_type']} ctr  K={cfg['pos_K']:.2f}")
        if h_opt:
            print(f"  Hedge option          : {h_opt['label']}")
            print(f"  Hedge contracts (fin) : {fin['w_h_end']:+.0f}")
        print(f"  Stock shares (final)  : {fin['w_s_end']:+.0f}")
        if pnl:
            print(f"  MTM P&L (final)       : ${fin['mtm_pnl']:+,.4f}")
            print(f"  MTM P&L max / min     : ${max(pnl):+,.4f}  /  ${min(pnl):+,.4f}")
        print(f"  Final net Delta       : {fin['net_delta']:+.4f}")
        print(f"  Final net Gamma       : {fin['net_gamma']:+.6f}\n")

        csv_path = f"{ASSET_NAME}_hedge_history.csv"
        pd.DataFrame(history).to_csv(csv_path, index=False)
        print(f"  History exported to {csv_path}")

        _sim_plot_pnl(history, cfg)
        _sim_plot_pnl_comparison(history, cfg)
        return history


    def _sim_plot_pnl(history: list, cfg: dict):
        try:
            import matplotlib.pyplot as plt
            import matplotlib.dates as mdates
            from matplotlib.ticker import FuncFormatter
        except ImportError:
            print("  ATTENTION! matplotlib not installed:skipping plot."); return

        h_opt  = cfg["h_opt"]
        rows   = [r for r in history if r["mtm_pnl"] is not None]
        if not rows:
            return

        dates_p = [r["date"] for r in rows]
        pnl  = [r["mtm_pnl"]    for r in rows]
        nd   = [r["net_delta"]   for r in rows]
        ng   = [r["net_gamma"]   for r in rows]
        sig  = [r["sigma"] * 100 for r in rows]
        rebal = [r["date"] for r in history if r["rebalanced"]]

        fig, (ax1, ax2, ax3) = plt.subplots(
            3, 1, figsize=(13, 9), sharex=True,
            gridspec_kw={"height_ratios": [3, 2, 1.5]})
        fig.suptitle(
            f"Hedging P&L  ·  {cfg['ticker']}  ·  {cfg['mode'].upper()}\n"
            f"{cfg['pos_dir'].upper()} {cfg['pos_qty']} {cfg['opt_type']} "
            f"K={cfg['pos_K']:.2f}  |  "
            + (f"Hedge: {h_opt['label']}" if h_opt else "No hedge option"),
            fontsize=11, y=0.98)

        def _rb(ax):
            for rd in rebal:
                ax.axvline(rd, color="grey", lw=0.5, ls="--", alpha=0.4)

        ax1.axhline(0, color="black", lw=0.8, alpha=0.6); _rb(ax1)
        ax1.fill_between(dates_p, pnl, 0, where=[v >= 0 for v in pnl],
                         alpha=0.25, color="seagreen", label="Gain")
        ax1.fill_between(dates_p, pnl, 0, where=[v < 0 for v in pnl],
                         alpha=0.25, color="crimson", label="Loss")
        ax1.plot(dates_p, pnl, color="steelblue", lw=1.6, label="MTM P&L")
        ax1.set_ylabel("MTM P&L ($)", fontsize=9)
        ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))
        ax1.legend(fontsize=8, loc="upper left")
        ax1.grid(axis="y", ls=":", alpha=0.4)

        ax2.axhline(0, color="black", lw=0.8, alpha=0.6); _rb(ax2)
        ax2.plot(dates_p, nd, color="darkorange", lw=1.4, label="net Delta")
        ax2r = ax2.twinx()
        ax2r.plot(dates_p, ng, color="mediumpurple", lw=1.2, ls="--", label="net Gamma")
        ax2r.set_ylabel("net Gamma", fontsize=8, color="mediumpurple")
        ax2r.tick_params(axis="y", labelcolor="mediumpurple", labelsize=7)
        ax2.set_ylabel("net Delta (shares)", fontsize=9)
        l1, lab1 = ax2.get_legend_handles_labels()
        l2, lab2 = ax2r.get_legend_handles_labels()
        ax2.legend(l1 + l2, lab1 + lab2, fontsize=8, loc="upper left")
        ax2.grid(axis="y", ls=":", alpha=0.4)

        _rb(ax3)
        ax3.plot(dates_p, sig, color="teal", lw=1.2)
        ax3.fill_between(dates_p, sig, alpha=0.15, color="teal")
        ax3.set_ylabel("sigma (%)", fontsize=9)
        ax3.set_xlabel("Date", fontsize=9)
        ax3.grid(axis="y", ls=":", alpha=0.4)
        ax3.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        fig.autofmt_xdate(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()


    def _sim_plot_pnl_comparison(history: list, cfg: dict):
        try:
            import matplotlib.pyplot as plt
            import matplotlib.dates as mdates
            from matplotlib.ticker import FuncFormatter
        except ImportError:
            print("  ATTENTION! matplotlib not installed:skipping comparison plot."); return

        rows = [r for r in history
                if r["mtm_pnl"] is not None and r.get("unhedged_pnl") is not None]
        if not rows:
            return

        dates_p   = [r["date"]         for r in rows]
        hedged    = [r["mtm_pnl"]      for r in rows]
        unhedged  = [r["unhedged_pnl"] for r in rows]
        spots     = [r["spot"]          for r in rows]
        advantage = [h - u for h, u in zip(hedged, unhedged)]

        h_opt       = cfg["h_opt"]
        mode_label  = cfg["mode"].upper()
        pos_label   = (f"{cfg['pos_dir'].upper()} {cfg['pos_qty']} "
                       f"{cfg['opt_type']} K={cfg['pos_K']:.2f}")
        hedge_label = h_opt["label"] if h_opt else "stock only"

        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(13, 7), sharex=True,
            gridspec_kw={"height_ratios": [3, 1.5]})
        fig.suptitle(
            f"Hedged vs Unhedged P&L  ·  {cfg['ticker']}  ·  {mode_label}\n"
            f"{pos_label}  |  Hedge: {hedge_label}",
            fontsize=11, y=0.98)

        ax1.axhline(0, color="black", lw=0.8, alpha=0.5)
        ax1.fill_between(dates_p, hedged, unhedged,
                         where=[a >= 0 for a in advantage],
                         alpha=0.18, color="seagreen", label="Hedge advantage")
        ax1.fill_between(dates_p, hedged, unhedged,
                         where=[a < 0 for a in advantage],
                         alpha=0.18, color="crimson", label="Hedge cost")
        ax1.plot(dates_p, unhedged, color="crimson",   lw=1.4, ls="--", label="Unhedged P&L")
        ax1.plot(dates_p, hedged,   color="steelblue", lw=1.8, label=f"Hedged P&L ({mode_label})")
        ax1.annotate(f"${hedged[-1]:+,.0f}",   xy=(dates_p[-1], hedged[-1]),
                     xytext=(8, 0), textcoords="offset points",
                     fontsize=8, color="steelblue", va="center")
        ax1.annotate(f"${unhedged[-1]:+,.0f}", xy=(dates_p[-1], unhedged[-1]),
                     xytext=(8, 0), textcoords="offset points",
                     fontsize=8, color="crimson", va="center")
        ax1.set_ylabel("MTM P&L ($)", fontsize=9)
        ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))
        ax1.legend(fontsize=8, loc="upper left")
        ax1.grid(axis="y", ls=":", alpha=0.4)

        ax2.plot(dates_p, spots, color="goldenrod", lw=1.4, label="Spot price")
        ax2.axhline(cfg["pos_K"], color="red", lw=0.8, ls=":",
                    label=f"Strike K={cfg['pos_K']:.2f}")
        ax2.set_ylabel("Spot ($)", fontsize=9)
        ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))
        ax2.legend(fontsize=8, loc="upper left")
        ax2.grid(axis="y", ls=":", alpha=0.4)
        ax2.set_xlabel("Date", fontsize=9)
        ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        fig.autofmt_xdate(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()


    def _sim_fetch_history(ticker: str, lookback_days: int = 252,
                           vol_window: int = VOL_WINDOW) -> tuple:
        end   = pd.Timestamp.today()
        start = end - pd.Timedelta(days=lookback_days + 30)
        raw   = yf.download(ticker,
                            start=start.strftime("%Y-%m-%d"),
                            end=end.strftime("%Y-%m-%d"),
                            progress=False, auto_adjust=True)
        if raw.empty:
            raise ValueError(f"No data found for '{ticker}'.")
        closes     = raw["Close"].squeeze().dropna()
        prices_all = [float(x) for x in closes.values]
        dates_all  = list(closes.index)
        if len(prices_all) < vol_window + 2:
            raise ValueError(f"Not enough history (need >= {vol_window+2} bars).")
        return prices_all, dates_all


    print("\n" + "=" * 60)
    print(f"  HISTORICAL HEDGE SIMULATION  ·  {ASSET_NAME}")
    print(f"  (strike K={K:.5f}  DTE={tau}d reused from pricer)")
    print("=" * 60)

    _mode    = _ask("Hedging mode [delta / delta_gamma] (default delta_gamma): ",
                    str, default="delta_gamma", valid={"delta", "delta_gamma"})
    _mult    = _ask("Contract multiplier (default 100): ", int, default=100)
    _pos_dir = _ask("Position direction [long / short] (default short): ",
                    str, default="short", valid={"long", "short"})
    _pos_opt = _ask("Position option type [call / put] (default call): ",
                    str, default="call", valid={"call", "put"})
    _pos_qty = max(1, _ask("Number of contracts (default 10): ", int, default=10))
    _dte     = int(tau)
    _hedge_days = _ask(f"Trading days to simulate (default {_dte}): ", int, default=_dte)
    _rebal_n    = _ask("Rebalance every N trading days (default 1): ", int, default=1)
    _pos_sign   = 1 if _pos_dir == "long" else -1

    _h_opt = None
    if _mode == "delta_gamma":
        _ht_auto, _hd_auto = _sim_auto_hedge_type(_pos_dir, _pos_opt)
        print("-" * 60)
        print("  Second option (gamma hedge)")
        print(f"  Auto rule: {_hd_auto.upper()} {_ht_auto.upper()}  "
              f"({_pos_dir} {_pos_opt} -> {_hd_auto} {_ht_auto})")
        _h_type = _ask(f"Hedge type [call / put] (default {_ht_auto}): ",
                       str, default=_ht_auto, valid={"call", "put"})
        _h_dir  = _ask(f"Hedge direction [long / short] (default {_hd_auto}): ",
                       str, default=_hd_auto, valid={"long", "short"})
        _atm_default = round(S * math.exp((r - q) * (_dte / 365.0)), 2)
        _h_K = _ask(f"Hedge strike (default {_atm_default:.5f}, near-ATM fwd): ",
                    float, default=_atm_default)
        _h_opt = dict(opt_type=_h_type, dir=_h_dir,
                      sign=(1 if _h_dir == "long" else -1),
                      K=_h_K, label=f"{_h_dir} {_h_type} K={_h_K:.2f}")

    print(f"\n  Fetching price history for {ASSET_NAME} ", end="", flush=True)
    _prices_all, _dates_all = _sim_fetch_history(ASSET_NAME)
    print("  done")

    _vol_window = VOL_WINDOW
    _hedge_days = max(1, min(_hedge_days, len(_prices_all)))
    if len(_prices_all) - _hedge_days < _vol_window:
        _vol_window = max(len(_prices_all) - _hedge_days, 5)

    _cfg_sim = dict(
        mode=_mode, ticker=ASSET_NAME,
        prices_all=_prices_all, dates_all=_dates_all,
        risk_free=r, div_yield=q, sigma=sigma,
        opt_type=_pos_opt, pos_dir=_pos_dir, pos_sign=_pos_sign,
        pos_qty=_pos_qty, pos_K=K, dte=_dte,
        h_opt=_h_opt, hedge_days=_hedge_days, rebal_n=_rebal_n,
        multiplier=_mult, vol_window=_vol_window,
    )

    _sim_run_simulation(_cfg_sim)

